In [1]:
import sys
sys.path.insert(0, '..')

import time
import numpy as np
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.dense_retriever import DenseRetriever
from src.retrieval.sparse_retriever import SparseRetriever
from src.retrieval.reranker import get_reranker

dense = DenseRetriever()
sparse = SparseRetriever()
hybrid = HybridRetriever()
reranker = get_reranker()
print(' All retrievers loaded\n')

# ─── Side-by-side comparison ───
query = "How does retrieval augmented generation reduce hallucinations?"
print(f'Query: "{query}"\n')

methods = {
    ' Dense (Semantic)': dense.search(query, top_k=5),
    ' Sparse (BM25)': sparse.search(query, top_k=5),
    'Hybrid (RRF)': hybrid.retrieve(query, top_k=5),
}

for name, results in methods.items():
    print(f'{name}:')
    for i, r in enumerate(results[:3]):
        src = r.metadata.get('source', '?')[:15]
        print(f'  [{i+1}] score={r.score:.4f} | {src:<15} | {r.content[:55]}...')
    print()

# ─── Alpha sweep ───
test_queries = [
    "What is retrieval augmented generation?",
    "BM25 ranking term frequency",
    "How do transformers handle long sequences?",
    "RLHF reward model preference alignment",
    "What causes hallucinations in LLMs?",
    "Explain attention mechanism in neural networks",
]

alphas = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]

print('Alpha Sweep (dense weight):')
print(f'{"Alpha":<8} {"Avg Top-3":<12} {"Avg Top-1":<12}')
print('─' * 32)

best_alpha, best_score = 0.0, 0.0
for alpha in alphas:
    t3, t1 = [], []
    for q in test_queries:
        res = hybrid.retrieve(q, top_k=3, alpha=alpha)
        if res:
            t3.append(np.mean([r.score for r in res]))
            t1.append(res[0].score)
    avg3 = np.mean(t3) if t3 else 0
    avg1 = np.mean(t1) if t1 else 0
    mark = ' ◀' if avg3 > best_score else ''
    if avg3 > best_score:
        best_score, best_alpha = avg3, alpha
    print(f'{alpha:<8.1f} {avg3:<12.6f} {avg1:<12.6f}{mark}')

print(f'\n Optimal alpha: {best_alpha}')

# ─── Reranker impact ───
print(f'\n\nReranker Impact:')
print('─' * 50)
for q in test_queries[:3]:
    pre = hybrid.retrieve(q, top_k=10)
    if len(pre) < 3:
        continue
    post = reranker.rerank(q, pre, top_k=5)
    changed = pre[0].content[:50] != post[0].content[:50]
    print(f'  Q: {q[:50]}...')
    print(f'    Before top-1: {pre[0].content[:45]}... ({pre[0].score:.4f})')
    print(f'    After  top-1: {post[0].content[:45]}... ({post[0].score:.4f})')
    print(f'    Reordered: {"Yes" if changed else "No"}\n')

# ─── Latency benchmark ───
print('\nLatency Benchmarks:')
print(f'{"Method":<22} {"Avg ms":<10}')
print('─' * 32)

for name, fn in [
    ('Dense only', lambda q: dense.search(q, top_k=5)),
    ('Sparse only', lambda q: sparse.search(q, top_k=5)),
    ('Hybrid (RRF)', lambda q: hybrid.retrieve(q, top_k=5)),
    ('Hybrid + Rerank', lambda q: reranker.rerank(q, hybrid.retrieve(q, top_k=10), top_k=5)),
]:
    lats = []
    for q in test_queries:
        t0 = time.time()
        fn(q)
        lats.append((time.time() - t0) * 1000)
    print(f'{name:<22} {np.mean(lats):<10.1f}')

print('\n Retrieval experiments complete')

<frozen importlib._bootstrap>:488: Warning: Numpy built with MINGW-W64 on Windows 64 bits is experimental, and only available for 
testing. You are advised not to use it for production. 

CRASHES ARE TO BE EXPECTED - PLEASE REPORT THEM TO NUMPY DEVELOPERS


: 